In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [23]:
data = pd.read_csv('/content/drive/MyDrive/employee_data.csv')

In [24]:
data.head()

,EmployeeId,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,1,38,NaN,Travel_Frequently,1444,Human Resources,1,4,Other,1,...,2,80,1,7,2,3,6,2,1,2
1,2,37,1.0,Travel_Rarely,1141,Research & Development,11,2,Medical,1,...,1,80,0,15,2,1,1,0,0,0
2,3,51,1.0,Travel_Rarely,1323,Research & Development,4,4,Life Sciences,1,...,3,80,3,18,2,4,10,0,2,7
3,4,42,0.0,Travel_Frequently,555,Sales,26,3,Marketing,1,...,4,80,1,23,2,4,20,4,4,8
4,5,40,NaN,Travel_Rarely,1194,Research & Development,2,4,Medical,1,...,2,80,3,20,2,3,5,3,0,2


In [25]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   EmployeeId                1470 non-null   int64  
 1   Age                       1470 non-null   int64  
 2   Attrition                 1058 non-null   float64
 3   BusinessTravel            1470 non-null   object 
 4   DailyRate                 1470 non-null   int64  
 5   Department                1470 non-null   object 
 6   DistanceFromHome          1470 non-null   int64  
 7   Education                 1470 non-null   int64  
 8   EducationField            1470 non-null   object 
 9   EmployeeCount             1470 non-null   int64  
 10  EnvironmentSatisfaction   1470 non-null   int64  
 11  Gender                    1470 non-null   object 
 12  HourlyRate                1470 non-null   int64  
 13  JobInvolvement            1470 non-null   int64  
 14  JobLevel

In [26]:
data.isnull().sum().sum()

np.int64(412)

In [27]:
data.duplicated().sum()

np.int64(0)

In [28]:
data=data.drop(['EmployeeId', 'Over18', 'StandardHours'], axis=1)
data.shape

(1470, 32)

In [29]:
df_model=data.dropna(subset=['Attrition'])
df_unlabeled=data[data['Attrition'].isna()]

In [30]:
df_model=pd.get_dummies(df_model, drop_first=True)

In [31]:
# Create X
X = df_model.drop(['Attrition'], axis=1)
# Create target object and call it y
y = df_model['Attrition']

In [32]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
    )

# Check the shape of X_train and X_test
X_train.shape, X_test.shape

((846, 45), (212, 45))

In [33]:
# Instantiate the classifier
rfc = RandomForestClassifier(random_state=0)

# Fit the model
rfc.fit(X_train, y_train)

# Predict the Test set result
y_pred = rfc.predict(X_test)

In [34]:
print('Accuracy Score: {0:0.4f}'. format(accuracy_score(y_test, y_pred)))
print("======================================================")
print("Classification Report: ")
print("======================================================")
print(classification_report(y_test, y_pred))
print("======================================================")
print("Confusion Matrix: ")
print("======================================================")
print(confusion_matrix(y_test, y_pred))

Accuracy Score: 0.8774
Classification Report: 
              precision    recall  f1-score   support

         0.0       0.88      0.99      0.93       178
         1.0       0.83      0.29      0.43        34

    accuracy                           0.88       212
   macro avg       0.86      0.64      0.68       212
weighted avg       0.87      0.88      0.85       212

Confusion Matrix: 
[[176   2]
 [ 24  10]]


In [35]:
feature_importance = (
    pd.Series(rfc.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(10)
)

print(feature_importance)

MonthlyIncome              0.066436
DailyRate                  0.052957
TotalWorkingYears          0.051874
Age                        0.050389
MonthlyRate                0.048921
HourlyRate                 0.047592
OverTime_Yes               0.046992
DistanceFromHome           0.046066
YearsAtCompany             0.043661
EnvironmentSatisfaction    0.035578
dtype: float64


## **INTERPRETASI**
1. **Faktor finansial dominan** : Gaji sangat mempengaruhi keputusan *resign*

*   `MonthlyIncome`, `DailyRate`, `MonthlyRate`, `HourlyRate` → semua tinggi

* **Insight :** Karyawan dengan kompensasi yang kurang kompetitif cenderung memiliki risiko *attrition* yang lebih tinggi.

2. **Pengalaman kerja & umur** : Karyawan muda / kurang pengalaman lebih rentan resign

*   `TotalWorkingYears`, `Age`, `YearsAtCompany` → semua tinggi

* **Insight :** Karyawan dengan pengalaman kerja yang lebih rendah cenderung memiliki tingkat *attrition* yang lebih tinggi dibandingkan karyawan senior.

3. **OverTime (lembur)**

*  `OverTime_Yes` berpengaruh terhadap *attrition*

* **Insight :** Karyawan yang sering melakukan lembur memiliki kecenderungan lebih tinggi untuk mengalami *attrition*.

4. **Jarak rumah ke kantor** : Semakin jauh jaraknya, semakin berisiko *resign*

*  `DistanceFromHome` berpengaruh terhadap *attrition*

* **Insight :** Jarak tempat tinggal yang jauh dari kantor dapat meningkatkan kemungkinan karyawan untuk *resign*.

5. **Kepuasan lingkungan kerja**

*  `EnvironmentSatisfaction` masuk top 10

* **Insight :** Tingkat kepuasan terhadap lingkungan kerja juga berkontribusi terhadap keputusan karyawan untuk bertahan atau *resign*.





